# How to run sequential methods

In the previous tutorials, we have inferred the posterior using **amortized inference**. In **amortized inference**, we draw parameters from the prior, simulate the corresponding data, and then train a neural network to obtain the posterior. However, if one is interested in only one particular observation `x_o` sampling from the prior can be inefficient in the number of simulations because one is effectively learning a posterior estimate for all observations in the prior space. In this tutorial, we show how one can alleviate this issue by using **sequential methods** with `sbi`.

**Sequential methods** also starts by drawing parameters from the prior, simulating them, and training a neural network to estimate the posterior distribution. Afterwards, however, it continues inference in multiple rounds, focusing on a particular observation `x_o`. In each new round of inference, it draws samples from the obtained posterior distribution conditioned at `x_o` (instead of from the prior), simulates these, and trains the network again. This process can be repeated arbitrarily often to get increasingly good approximations to the true posterior distribution at `x_o`.

Running multi-round inference can be more efficient in the number of simulations, but it will lead to the posterior no longer being amortized (i.e. it will be accurate only for a specific observation `x_o`, not for any `x`).


## Main syntax


```python
inference = NPE(prior)
proposal = prior

for _ in range(num_rounds):
    theta = proposal.sample((100,))
    x = simulate(theta)

    # In `SNLE` and `SNRE`, you should not pass the `proposal` to `.append_simulations()`.
    density_estimator = inference.append_simulations(
        theta, x, proposal=proposal
    ).train()
    posterior = inference.build_posterior(density_estimator)
    proposal = posterior.set_default_x(x_o)
```


## Tracking convergence across rounds

A common question when running sequential inference is: how many rounds are enough? `sbi` provides [`SequentialConvergenceTracker`](https://sbi.readthedocs.io/en/latest/api_reference.html#sbi.diagnostics.SequentialConvergenceTracker) to help answer this by reporting three quantities at each round:

- **compression** $\mathrm{KL}(q_r \| \pi)$: how far the current estimate has travelled from the prior.
- **increment** $\mathrm{KL}(q_r \| q_{r-1})$: how much the last round moved the estimate.
- **ratio** increment / compression: the fraction of the distance from the prior contributed by the last round.

When the increment and ratio become small and the compression is stable, further rounds are unlikely to improve the estimate meaningfully. The tracker is a diagnostic, not an automatic stopping rule: it measures the self-consistency of the sequence, not the distance to the true posterior. To check correctness, use [`run_sbc`](https://sbi.readthedocs.io/en/latest/api_reference.html#sbi.diagnostics.run_sbc) or [`run_tarp`](https://sbi.readthedocs.io/en/latest/api_reference.html#sbi.diagnostics.run_tarp).

The tracker works with any sequential method (NPE-C, NPE-A, TSNPE, etc.) and fits into the loop with two extra lines:


```python
from sbi.diagnostics import SequentialConvergenceTracker

inference = NPE(prior)
tracker = SequentialConvergenceTracker(prior, x_o, num_samples=2000)
proposal = prior

for _ in range(num_rounds):
    theta = proposal.sample((100,))
    x = simulate(theta)
    inference.append_simulations(theta, x, proposal=proposal).train()
    posterior = inference.build_posterior().set_default_x(x_o)

    tracker.update(posterior)   # returns a dict with per-round diagnostics
    print(tracker.ratios[-1])    # fraction of total compression from this round

    proposal = posterior
```


After the loop, `tracker.history` holds the full record (round, compression, increment, ratio, and standard errors). The convenience properties `.compressions`, `.increments` and `.ratios` return the values as lists for plotting.

The tracker also flags rounds where the estimate is indistinguishable from the prior (`uninformative=True`, ratio reported as `NaN`), which can happen with weakly informative observations or in very early rounds before training has moved the estimate.


## Example

You can find a full example and more explanation in the tutorial [here](https://sbi.readthedocs.io/en/latest/advanced_tutorials/02_multiround_inference.html).
